# Consolidação dos testes de usabilidade

Consolida as sessões de usabilidade a partir dos dados anonimizados em `docs/usabilidade/dados/`, preenchidos conforme
[`docs/usabilidade/README.md`](../../docs/usabilidade/README.md).

O notebook:

1. valida os valores de cada planilha antes de qualquer cálculo;
2. separa sessões **realizadas**, **bloqueadas** e **não executadas**;
3. apura cada cenário nos estados *concluído*, *concluído com dificuldade*, *falhou* e *não executado*;
4. calcula o SUS de cada participante e o descreve por mediana e amplitude, sem meta de aprovação;
5. lista as ocorrências por severidade e os defeitos críticos sem tratamento ou sem reteste.

Executar a partir de `src/notebooks`. A variável de ambiente `USABILIDADE_DADOS` aponta para outro diretório de dados,
o que permite testar o notebook sem tocar nos arquivos versionados.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DADOS = Path(os.environ.get('USABILIDADE_DADOS', '../../docs/usabilidade/dados'))
FIGURAS = Path(os.environ.get('USABILIDADE_FIGURAS', '../../docs/img'))

AMOSTRA_MINIMA = 5
RESULTADOS = ['sucesso', 'dificuldade', 'nao_conseguiu', 'nao_testado']
SEVERIDADES = ['alta', 'media', 'baixa', 'cosmetica']
STATUS_SESSAO = ['realizada', 'bloqueada', 'nao_executada']
FAMILIARIDADE = ['baixa', 'media', 'alta']
FAIXAS_ETARIAS = ['18-29', '30-39', '40-49', '50-59', '60+']

participantes = pd.read_csv(DADOS / 'participantes.csv', dtype=str).fillna('')
registro = pd.read_csv(DADOS / 'registro.csv', dtype=str).fillna('')
ocorrencias = pd.read_csv(DADOS / 'ocorrencias.csv', dtype=str).fillna('')
sus = pd.read_csv(DADOS / 'sus.csv', dtype=str).fillna('')
print(f'dados lidos de {DADOS}')

## 1. Validação

Um valor fora do vocabulário do README interrompe a consolidação: um resultado digitado errado não pode virar
estatística silenciosamente. Linhas de participantes cuja sessão não aconteceu podem ficar em branco.

In [ ]:
problemas = []

def conferir(tabela, nome, coluna, permitidos, filtro=None):
    linhas = tabela if filtro is None else tabela[filtro]
    for indice, valor in linhas[coluna].items():
        if valor not in permitidos:
            problemas.append(f'{nome}, linha {indice + 2}, {coluna}: {valor!r} fora de {permitidos}')

sessoes_com_status = participantes['status_sessao'] != ''
conferir(participantes, 'participantes.csv', 'status_sessao', STATUS_SESSAO, sessoes_com_status)
realizadas = set(participantes.loc[participantes['status_sessao'] == 'realizada', 'participante'])
p_realizadas = participantes['participante'].isin(realizadas)
conferir(participantes, 'participantes.csv', 'externo_a_turma', ['sim'], p_realizadas)
conferir(participantes, 'participantes.csv', 'familiaridade_tecnologica', FAMILIARIDADE, p_realizadas)
conferir(participantes, 'participantes.csv', 'familiaridade_agentes', FAMILIARIDADE, p_realizadas)
conferir(participantes, 'participantes.csv', 'faixa_etaria', FAIXAS_ETARIAS, p_realizadas)
for indice, linha in participantes[p_realizadas].iterrows():
    if not linha['commit_testado']:
        problemas.append(f"participantes.csv, linha {indice + 2}: sessão realizada sem commit_testado")
    if not linha['moderador']:
        problemas.append(f"participantes.csv, linha {indice + 2}: sessão realizada sem moderador")
for indice, linha in participantes[participantes['status_sessao'] == 'bloqueada'].iterrows():
    if not linha['motivo_status']:
        problemas.append(f"participantes.csv, linha {indice + 2}: sessão bloqueada sem motivo")

r_realizadas = registro['participante'].isin(realizadas)
for coluna in ['resultado_geral', 'etapa_1', 'etapa_2', 'etapa_3', 'etapa_4', 'etapa_5']:
    conferir(registro, 'registro.csv', coluna, RESULTADOS, r_realizadas)
conferir(registro, 'registro.csv', 'dica_dada', ['sim', 'nao'], r_realizadas)
for indice, linha in registro[r_realizadas].iterrows():
    etapas = [linha[f'etapa_{n}'] for n in range(1, 6)]
    if linha['resultado_geral'] == 'nao_testado' and not linha['observacoes']:
        problemas.append(f'registro.csv, linha {indice + 2}: nao_testado sem a dependência nomeada nas observações')
    if linha['dica_dada'] == 'sim' and 'nao_conseguiu' not in etapas:
        problemas.append(f'registro.csv, linha {indice + 2}: dica dada sem etapa registrada como nao_conseguiu')

conferir(ocorrencias, 'ocorrencias.csv', 'severidade', SEVERIDADES)
conferir(ocorrencias, 'ocorrencias.csv', 'defeito_critico', ['sim', 'nao'])
fora = sorted(set(ocorrencias['participante']) - realizadas - {''})
if fora:
    problemas.append(f'ocorrencias.csv: participantes sem sessão realizada: {fora}')

itens = [f'q{n}' for n in range(1, 11)]
s_realizadas = sus['participante'].isin(realizadas)
for indice, linha in sus[s_realizadas].iterrows():
    for item in itens:
        if linha[item] not in {'', '1', '2', '3', '4', '5'}:
            problemas.append(f'sus.csv, linha {indice + 2}, {item}: {linha[item]!r} fora de 1 a 5')

if problemas:
    print('\n'.join(problemas))
    raise ValueError(f'{len(problemas)} problema(s) nos dados; corrigir antes de consolidar')
print('dados válidos')

## 2. Sessões

O planejamento exige ao menos cinco participantes externos à turma. Abaixo disso, a consolidação continua sendo
calculada, mas a amostra é declarada insuficiente.

In [ ]:
contagem = participantes['status_sessao'].replace('', 'sem registro').value_counts()
print(contagem.to_string())
amostra_suficiente = len(realizadas) >= AMOSTRA_MINIMA
print()
print(f'sessões realizadas: {len(realizadas)} de {AMOSTRA_MINIMA} exigidas '
      f'— amostra {"suficiente" if amostra_suficiente else "INSUFICIENTE"}')

perfis = participantes[p_realizadas][['participante', 'vaga_perfil', 'familiaridade_tecnologica',
                                     'familiaridade_agentes', 'faixa_etaria', 'commit_testado']]
commits = set(perfis['commit_testado'])
if len(commits) > 1:
    print(f'ATENÇÃO: sessões em versões diferentes: {sorted(commits)}')
vagas_vazias = participantes.loc[~p_realizadas & (participantes['vaga_perfil'] != 'reserva'), 'vaga_perfil'].tolist()
if vagas_vazias:
    print(f'vagas de perfil sem sessão realizada: {vagas_vazias}')
familiaridades = set(perfis['familiaridade_agentes'])
if realizadas and not {'baixa', 'alta'} <= familiaridades:
    print(f'ATENÇÃO: a amostra não reúne familiaridade baixa e alta com agentes: {sorted(familiaridades)}')
perfis

## 3. Resultado por cenário

Estados apurados: *concluído* (`sucesso`), *concluído com dificuldade* (`dificuldade`), *falhou*
(`nao_conseguiu`) e *não executado* (`nao_testado`). Sessões bloqueadas não entram aqui; aparecem na seção 2.

In [ ]:
ESTADOS = {'sucesso': 'concluído', 'dificuldade': 'concluído com dificuldade',
           'nao_conseguiu': 'falhou', 'nao_testado': 'não executado'}
executado = registro[r_realizadas].copy()
executado['estado'] = executado['resultado_geral'].map(ESTADOS)
por_cenario = (executado.pivot_table(index=['cenario', 'tarefa'], columns='estado', values='participante',
                                     aggfunc='count', fill_value=0)
               .reindex(columns=list(ESTADOS.values()), fill_value=0))
por_cenario

In [ ]:
if len(executado):
    FIGURAS.mkdir(parents=True, exist_ok=True)
    cores = ['#2e7d32', '#f9a825', '#c62828', '#9e9e9e']
    tabela = por_cenario.droplevel(0)
    eixo = tabela.plot(kind='barh', stacked=True, color=cores, figsize=(10, 4.8))
    eixo.invert_yaxis()
    eixo.set_xlabel('participantes')
    eixo.set_ylabel('')
    eixo.set_title(f'Resultado por cenário ({len(realizadas)} sessões realizadas)')
    eixo.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.savefig(FIGURAS / 'usabilidade-resultado-por-cenario.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Etapas com mais dificuldade

As etapas em que mais participantes hesitaram, falharam ou precisaram de dica. É a leitura que aponta o que
corrigir; o SUS, sozinho, não aponta.

In [ ]:
linhas = []
for _, linha in executado.iterrows():
    for n in range(1, 6):
        linhas.append({'cenario': linha['cenario'], 'tarefa': linha['tarefa'], 'etapa': n,
                       'resultado': linha[f'etapa_{n}']})
etapas = pd.DataFrame(linhas, columns=['cenario', 'tarefa', 'etapa', 'resultado'])
testadas = etapas[etapas['resultado'] != 'nao_testado']
if len(testadas):
    resumo = (testadas.assign(problema=testadas['resultado'].isin(['dificuldade', 'nao_conseguiu']))
              .groupby(['cenario', 'tarefa', 'etapa'])['problema'].agg(['sum', 'count'])
              .rename(columns={'sum': 'com_problema', 'count': 'participantes'}))
    resumo = resumo[resumo['com_problema'] > 0].sort_values('com_problema', ascending=False)
    display(resumo.head(10))
dicas = executado[executado['dica_dada'] == 'sim'][['participante', 'cenario', 'tarefa']]
print(f'dicas dadas: {len(dicas)}')
dicas

## 5. SUS

Dado descritivo, sem meta de aprovação (ver [`questionario-sus.md`](../../docs/usabilidade/questionario-sus.md)).
Itens ímpares contribuem com resposta − 1; pares, com 5 − resposta; a soma multiplicada por 2,5 dá o SUS de 0 a 100.
Resposta em branco invalida o SUS do participante.

In [ ]:
def pontuar(linha):
    if any(linha[item] == '' for item in itens):
        return None
    valores = [int(linha[item]) for item in itens]
    soma = sum(v - 1 if n % 2 == 1 else 5 - v for n, v in enumerate(valores, start=1))
    return soma * 2.5

respostas = sus[s_realizadas].copy()
respostas['sus'] = respostas.apply(pontuar, axis=1)
invalidos = respostas.loc[respostas['sus'].isna(), 'participante'].tolist()
validos = respostas.dropna(subset=['sus']).merge(perfis, on='participante')
print(f'SUS válidos: {len(validos)}; inválidos (resposta em branco): {invalidos or "nenhum"}')
if len(validos):
    print(f"mediana {validos['sus'].median():.1f} | amplitude {validos['sus'].min():.1f} a {validos['sus'].max():.1f} "
          f"| média {validos['sus'].mean():.1f} com n = {len(validos)}")
validos[['participante', 'vaga_perfil', 'familiaridade_agentes', 'sus']].sort_values('participante')

## 6. Ocorrências e defeitos críticos

Todo defeito crítico precisa estar corrigido ou ter impedimento declarado, e precisa de reteste.

In [ ]:
if len(ocorrencias):
    por_severidade = (ocorrencias.pivot_table(index='cenario', columns='severidade', values='id',
                                              aggfunc='count', fill_value=0)
                      .reindex(columns=SEVERIDADES, fill_value=0))
    display(por_severidade)
criticos = ocorrencias[ocorrencias['defeito_critico'] == 'sim']
pendentes = criticos[(criticos['tratamento'] == '') | (~criticos['reteste'].isin(['aprovado', 'reprovado']))]
altas_sem_avaliacao = ocorrencias[(ocorrencias['severidade'] == 'alta') & (ocorrencias['defeito_critico'] == '')]
print(f'ocorrências: {len(ocorrencias)} | defeitos críticos: {len(criticos)} | '
      f'críticos sem tratamento ou reteste concluído: {len(pendentes)}')
if len(altas_sem_avaliacao):
    print(f'ATENÇÃO: {len(altas_sem_avaliacao)} ocorrência(s) alta(s) sem avaliação de defeito crítico')
criticos[['id', 'cenario', 'participante', 'resumo', 'tratamento', 'reteste']]

## 7. Comentários abertos

Respostas às três perguntas do encerramento, já anonimizadas, para a síntese qualitativa da Seção 9.3.2.

In [ ]:
for _, linha in respostas[respostas['comentario_aberto'] != ''].iterrows():
    print(f"{linha['participante']}: {linha['comentario_aberto']}\n")